In [15]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoProcessor, AutoModelForCausalLM

class GatedCrossAttentionLayer(nn.Module):
   def __init__(self, hidden_dim, num_heads=8):
       super().__init__()
       self.cross_attn = nn.MultiheadAttention(
           embed_dim=hidden_dim, num_heads=num_heads, batch_first=True
       )
       self.ffn = nn.Sequential(
           nn.Linear(hidden_dim, hidden_dim * 4),
           nn.GELU(),
           nn.Linear(hidden_dim * 4, hidden_dim),
       )
       self.attn_gate = nn.Parameter(torch.zeros(1))  # init at 0
       self.ffn_gate = nn.Parameter(torch.zeros(1))   # init at 0

   def forward(self, text_hidden, image_features):
       # 1. Gated cross-attention
       attn_out, _ = self.cross_attn(
           query=text_hidden, key=image_features, value=image_features
       )
       text_hidden = text_hidden + self.attn_gate.tanh() * attn_out
       # 2. Gated feed-forward
       text_hidden = text_hidden + self.ffn_gate.tanh() * self.ffn(text_hidden)

       return text_hidden

class CrossAttentionVLM(nn.Module):
    def __init__(self, vision_encoder_ckpt, language_model_ckpt, tokenizer,
                 vision_dim=768, num_visual_tokens=64, cross_attn_every_n=4):
        super().__init__()

        # --- Vision side (frozen, same as unified sequence version) ---
        self.vision_encoder = AutoModel.from_pretrained(vision_encoder_ckpt).vision_model
        self.vision_encoder.requires_grad_(False)
        self.vision_processor = AutoProcessor.from_pretrained(vision_encoder_ckpt)

        # Perceiver Resampler: compress to fixed number of visual tokens
        self.learned_queries = nn.Parameter(torch.randn(1, num_visual_tokens, vision_dim))
        self.resampler_attn = nn.MultiheadAttention(
            embed_dim=vision_dim, num_heads=8, batch_first=True
        )
        self.resampler_proj = nn.Linear(vision_dim, self._get_llm_dim(language_model_ckpt))

        # --- Language side (frozen) ---
        self.tokenizer = tokenizer
        self.llm = AutoModelForCausalLM.from_pretrained(language_model_ckpt)
        self.llm.requires_grad_(False)

        llm_dim = self.llm.config.hidden_size

        # --- Gated cross-attention layers (the only trained part) ---
        self.cross_attn_layers = nn.ModuleDict()
        for i in range(0, self.llm.config.num_hidden_layers, cross_attn_every_n):
            self.cross_attn_layers[str(i)] = GatedCrossAttentionLayer(llm_dim)

    @staticmethod
    def _get_llm_dim(ckpt):
        from transformers import AutoConfig
        return AutoConfig.from_pretrained(ckpt).hidden_size

    def encode_image(self, image):
        processed = self.vision_processor(images=[image], return_tensors="pt")
        processed = processed.to(next(self.vision_encoder.parameters()).device)
        with torch.no_grad():
            vision_features = self.vision_encoder(**processed).last_hidden_state
        queries = self.learned_queries.expand(vision_features.size(0), -1, -1)
        resampled, _ = self.resampler_attn(
            query=queries, key=vision_features, value=vision_features
        )
        return self.resampler_proj(resampled)

    def forward(self, input_ids, image):
        image_features = self.encode_image(image)

        # Walk through frozen LLM layers, injecting cross-attention
        hidden = self.llm.model.embed_tokens(input_ids)
        for i, layer in enumerate(self.llm.model.layers):
            if str(i) in self.cross_attn_layers:
                hidden = self.cross_attn_layers[str(i)](hidden, image_features)
            hidden = layer(hidden)[0]

        hidden = self.llm.model.norm(hidden)
        logits = self.llm.lm_head(hidden)

        # Loss (same as the unified sequence version from chapter 3)
        labels = input_ids.clone()
        labels[labels == self.tokenizer.image_token_id] = self.tokenizer.pad_token_id
        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()
        loss = nn.functional.cross_entropy(
            shift_logits.reshape(-1, shift_logits.size(-1)),
            shift_labels.reshape(-1),
            ignore_index=self.tokenizer.pad_token_id,
        )
        return logits, loss

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer
import torch.optim as optim
import time
import json
import numpy as np

vision_encoder_ckpt = "google/siglip2-base-patch16-256"
language_model_ckpt = "HuggingFaceTB/SmolLM2-135M-Instruct"
dataset = load_dataset("HuggingFaceM4/FineVisionMax", split="train", streaming=True)
tokenizer = AutoTokenizer.from_pretrained(language_model_ckpt, extra_special_tokens={"image_token": "<|image|>"})
vlm = CrossAttentionVLM(vision_encoder_ckpt, language_model_ckpt, tokenizer).to("cuda")
optimizer = optim.AdamW(vlm.parameters(), lr=1e-4)

def tokenize_sample(sample):
    messages_list = []
    for data_dict in sample['texts']:
        messages_list.append({'role': 'user', 'content': data_dict['user']})
        messages_list.append({'role': 'assistant', 'content': data_dict['assistant']})
    messages_list[0]['content'] = tokenizer.image_token*256 + messages_list[0]['content']  # add image tokens
    tokens = tokenizer.apply_chat_template(messages_list, tokenize=True)
    return tokens['input_ids']


step = 0
losses = []
max_length = 2048
curr_sample = []
curr_image = []
batch = []
image_batch = []
max_batch_size = 4
batch_complete = False
start = time.time()
for sample in dataset:
    if len(sample['images']) != 1:
      continue
    tokens = tokenize_sample(sample)
    if len(tokens) > max_length:
        continue  # Skip samples that are too long
    if len(curr_sample) + len(tokens) < max_length:
        curr_sample.extend(tokens)
        curr_image.extend([sample['images'][0].convert('RGB')])
    else:
        batch.append(torch.nn.functional.pad(torch.Tensor(curr_sample), (max_length - len(curr_sample),0), 'constant', tokenizer.pad_token_id))
        image_batch.extend(curr_image)
        if len(batch) == max_batch_size:
          batch_complete = True # We've reached the max batch size, so we can continue
        curr_sample = tokens
        curr_image = [sample['images'][0].convert('RGB')]
    if batch_complete:
      step += 1
      if step%100 == 0:
        print(f"step {step} | loss {loss.item():.4f} | smoothed loss: {np.mean(losses[-10:]):.4f}")
        print(f"The last 100 steps took: {time.time()-start:.2f} seconds.")
        start = time.time()

      optimizer.zero_grad()
      logits, loss = vlm(torch.stack(batch).long().cuda(), image_batch)
      losses.append(loss.item())

      loss.backward()
      optimizer.step()

      batch_complete = False
      batch = []
      image_batch = []
      if step > 5000:
        break

json.dump([round(l, 4) for l in losses], open("chapter_6_batched_loss.json", "w"))

Resolving data files:   0%|          | 0/10000 [00:00<?, ?it/s]